In [ ]:
import os
#os.chdir('/Users/asahoo/laurent_repos/PASTIS/pastis/')
os.chdir('/Users/asahoo/repos/PASTIS/pastis/')
import sys
sys.path.append('/Users/asahoo/repos/PASTIS/pastis/')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from astropy.io import fits
import astropy.units as u
import hcipy as hc
import matplotlib as mpl
import util_pastis as util
from config import CONFIG_INI

from hockeystick_contrast_curve import hockeystick_luvoir
from e2e_simulators.luvoir_imaging_onephot import LuvoirAPLC

In [ ]:
nseg = 120            #for luvoir?
wvln = 638e-9

datadir = '/Users/asahoo/laurent_repos/PASTIS/LUVOIR_delivery_May2019/' #confused what this directory actually does?
aper_path = 'inputs/TelAp_LUVOIR_gap_pad01_bw_ovsamp04_N1000.fits'
aper_ind_path = 'inputs/TelAp_LUVOIR_gap_pad01_bw_ovsamp04_N1000_indexed.fits'
aper_read = hc.read_fits(os.path.join(datadir, aper_path))
aper_ind_read = hc.read_fits(os.path.join(datadir, aper_ind_path))

pupil_grid = hc.make_pupil_grid(dims=aper_ind_read.shape[0], diameter=15)
aper = hc.Field(aper_read.ravel(), pupil_grid)
aper_ind = hc.Field(aper_ind_read.ravel(), pupil_grid)
lyot_stop_path_in_optics = 'inputs/LS_LUVOIR_ID0120_OD0982_no_struts_gy_ovsamp4_N1000.fits'

calibration_aberration = 1.
valid_range_lower = -4
valid_range_upper = 4

sampling = 4.

wf_aper = hc.Wavefront(aper, wvln)

# Load segment positions from fits header
hdr = fits.getheader(os.path.join(datadir, aper_ind_path))

poslist = []
for i in range(nseg):
    segname = 'SEG' + str(i+1)
    xin = hdr[segname + '_X']
    yin = hdr[segname + '_Y']
    poslist.append((xin, yin))
    
poslist = np.transpose(np.array(poslist))
seg_pos = hc.CartesianGrid(hc.UnstructuredCoords(poslist))
eunit = 1e-9

In [ ]:
design='small'
optics_input = CONFIG_INI.get('LUVOIR', 'optics_path')
luvoir = LuvoirAPLC(optics_input, design, sampling)

In [ ]:
segment = hc.hexagonal_aperture(luvoir.segment_circum_diameter, np.pi / 2)
segment_sampled = hc.evaluate_supersampled(segment,luvoir.pupil_grid, 1)
aper2, segs2 = hc.make_segmented_aperture(segment,luvoir.seg_pos, segment_transmissions=1, return_segments=True)
luvoir_segmented_pattern = hc.evaluate_supersampled(aper2, luvoir.pupil_grid, 1)
seg_evaluated = []
for seg_tmp in segs2:
    tmp_evaluated = hc.evaluate_supersampled(seg_tmp, luvoir.pupil_grid, 1)
    seg_evaluated.append(tmp_evaluated)

In [ ]:
print(type(luvoir_segmented_pattern))
#print(type(seg_evaluated), "\n", seg_evaluated[0])
#print(len(seg_evaluated), seg_evaluated[0].shape)
#plt.imshow((np.abs(np.reshape(seg_evaluated[10], [1000,1000]))))
#plt.colorbar();
hc.imshow_field(luvoir_segmented_pattern)

In [ ]:
run_choice = CONFIG_INI.get('numerical', 'current_analysis')
coro_design = CONFIG_INI.get('LUVOIR', 'coronagraph_size')

result_dir = os.path.join(CONFIG_INI.get('local', 'local_data_path'), run_choice, 'results')
matrix_dir = os.path.join(CONFIG_INI.get('local', 'local_data_path'), run_choice, 'matrix_numerical')


In [ ]:
hockeystick_luvoir(apodizer_choice=coro_design, matrixdir=matrix_dir, resultdir=result_dir, range_points=30, no_realizations=5)

In [ ]:
from hcipy.optics.segmented_mirror import SegmentedMirror

In [ ]:
conda env list 